# Generative AI Applications Project
* **Gene Foxwell**

## Overview

This project looks at how a Generative Pretrained Transformer Model (GPT) can be used to solve Reinforcement Learning Problems. The idea is based on the Decision Transformers paper (PAPER). We will use the Human Relocate Data from Minari's DR4RL section. Further information on the Relocate data can be found here (PAPER). The relocate problem was originally introduced here (PAPER). 

The basic problem is to train virtual 24 DoF robotic hand to pick up a ball from one location and move it to another.

We'll attack this problem in the following sequence:

* Build a GPT Model that we can use with a Decision Transformer.
* We'll build an implementation of the Decision Transformer Algorithm.
* We'll train the algorithm on the Human generated data for the Relocate the problem.
* Summary of results.

Let's get started...

## Transformer Model

We'll build our transformer model based on the architecture described in (ATTENTION PAPER) as well as examples from Udacity's course work.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import minari
import gymnasium as gym
import gymnasium_robotics

AdroitHandRelocateDense-v1, AdroitHandHammerDense-v1, AdroitHandDoorDense-v1 environment's reward functions were updated in v1.2.1 without an environment version update. Therefore, use gymnasium-robotics==1.2.0 for v1 reproducibility or use v2 in gymnasium-robotics>=1.4.3. See https://github.com/Farama-Foundation/Gymnasium-Robotics/pull/220 for more details


In [2]:
# Use CUDA if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
class AttentionHead(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.Q_weights = nn.Linear(
            config["embed_dim"], config["head_size"], config["use_bias"]
        )
        self.K_weights = nn.Linear(
            config["embed_dim"], config["head_size"], config["use_bias"]
        )
        self.V_weights = nn.Linear(
            config["embed_dim"], config["head_size"], config["use_bias"]
        )

        self.dropout = nn.Dropout(config["dropout_rate"])

        casual_attention_mask = torch.tril(
            torch.ones(config["context_size"], config["context_size"])
        )
        self.register_buffer("casual_attention_mask", casual_attention_mask)

    def forward(self, input):  # (B, C, embedding_dim)
        batch_size, tokens_num, embedding_dim = input.shape
        Q = self.Q_weights(input)  # (B, C, head_size)
        K = self.K_weights(input)  # (B, C, head_size)
        V = self.V_weights(input)  # (B, C, head_size)

        # Matrix Multiplay Q x K transpose to get the dot product of the query vectors with the key vectors
        attention_scores = Q @ K.transpose(1, 2)  # (B, C, C)

        # scale attention scores, scalled by square root of the dimensionality of the key vectors
        attention_scores = attention_scores / (K.shape[-1] ** 0.5)
        
        # mask the attention scores
        attention_scores = attention_scores.masked_fill(
            self.casual_attention_mask[:tokens_num, :tokens_num] == 0, -torch.inf
        )

        # calculate softmax values
        attention_scores = torch.softmax(attention_scores, dim=-1)

        # apply dropout for regularization
        attention_scores = self.dropout(attention_scores)

        # multiply attention scores by the value function.
        return attention_scores @ V  # (B, C, head_size)

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()

        # initialize the individual AttentionHead objects
        heads_list = [AttentionHead(config) for _ in range(config["heads_num"])]
        self.heads = nn.ModuleList(heads_list)

        # Feedforward connection for after the attention heads
        self.linear = nn.Linear(config["embed_dim"], config["embed_dim"])

        # Dropout regularization.
        self.dropout = nn.Dropout(config["dropout_rate"])

    def forward(self, input):
        # execute heads in ||
        heads_outputs = [head(input) for head in self.heads]

        # concatenate the outputs into a single tensor
        scores_change = torch.cat(heads_outputs, dim=-1)

        # run the results through a feed forward network.
        scores_change = self.linear(scores_change)

        # regularization and return results
        return self.dropout(scores_change)

In [5]:
class FeedForward(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.linear_layers = nn.Sequential(
            nn.Linear(config["embed_dim"], config["embed_dim"] * 4),
            nn.GELU(),
            nn.Linear(config["embed_dim"] * 4, config["embed_dim"]),
            nn.Dropout(config["dropout_rate"]),
        )

    def forward(self, input):
        return self.linear_layers(input)

In [6]:
class Block(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.multi_head = MultiHeadAttention(config)
        self.layer_norm_1 = nn.LayerNorm(config["embed_dim"])

        self.feed_forward = FeedForward(config)
        self.layer_norm_2 = nn.LayerNorm(config["embed_dim"])

    def forward(self, input):
        residual = input
        x = self.multi_head(self.layer_norm_1(input))
        x = x + residual

        residual = x
        x = self.feed_forward(self.layer_norm_2(x))
        return x + residual

In [7]:
class TransformerModel(nn.Module):
    def __init__(self, config):
        super().__init__()

        blocks = [Block(config) for _ in range(config["layers_num"])]
        self.layers = nn.Sequential(*blocks)
        self.layer_norm = nn.LayerNorm(config["embed_dim"])


    def forward(self, input_embeddings):
        """
        Forward step for the transformer model. The DecisionTransformer already handles embeddings and positional encodings.
        We are simply making predictions using the AttentionHeads and returning the final hidden layer.
        """
        
        # Pass the embeddings through the stacked Transformer blocks
        x = self.layers(input_embeddings)
        
        # Apply the final layer normalization
        return self.layer_norm(x)

## Decision Transformer

We'll base our implementation of the Decision Transformer based on the original papers github repo found here: (https://github.com/kzl/decision-transformer)

In [ ]:
class DecisionTransformer(nn.Module):
    def __init__(self, config):

        # size of the hidden layer
        self.hidden_size = config["embed_dim"]

        # maximum number of timetimes in an episode
        max_ep_length = config["max_ep_length"]
        
        # dimension of the state space
        self.state_dim = config["state_dim"]

        # dimension of the action space.
        self.action_dim = config["action_dim"]

        # embedding layers for timestamps, returns, states, and actions (t,r,s,a)
        # remember, at each time timestamp, we have a triple:
        #   r = expected return
        #   s = state
        #   a = action
        self.embed_timestep = nn.Embedding(max_ep_len, self.hidden_size)
        self.embed_return = nn.Linear(1, self.hidden_size)
        self.embed_state = nn.Linear(self.state_dim, self.hidden_size)
        self.embed_action = nn.Linear(self.action_dim, self.hidden_size)

        # Normalization of the embedding layers
        self.embed_ln = nn.LayerNorm(self.hidden_size)
        
        # action prediction layer
        self.predict_action = nn.Linear(self.hidden_size, self.action_dim)

        # create an instance of the transformer model
        # this will be used to generate the next steps in the sequence.
        self.transformer = TransformerModel(config)

    def forward(self, states, actions, rewards, returns_to_go, timesteps):
        """
        Generate the next action using the previous Returns, States, Actions, and Timestamps.
        """
        batch_size, seq_length = states.shape[0], states.shape[1]
        
        pos_embedding = self.embed_timestep(timesteps)
        
        state_embeddings = self.embed_state(states) + pos_embedding
        action_embeddings = self.embed_action(actions) + pos_embedding
        returns_embeddings = self.embed_return(rewards) + pos_embedding
        
        stacked_inputs = torch.stack(
            (returns_embeddings, state_embeddings, action_embeddings), dim=1
        ).permute(0, 2, 1, 3).reshape(batch_size, 3*seq_length, self.hidden_size)
        
        input_embeddings = self.embed_ln(stacked_inputs)
        
        hidden_states = self.transformer(input_embeddings)
        
        state_representations = hidden_states[:, 1::3, :] 
        
        action_logits = self.predict_action(state_representations)
        return torch.tanh(action_preds)

## ADROIT Hand Relocate Problem

We are focusing on the ADROIT Relocation problem. In this problem we will drain a robotic hand to find a ball, pick it up, and then drop it off at another location. We will train this hand using human generated data provided by Minari. Our agent will be based on the DecisionTransformer approach above - autoregressively outputing the policy needed to solve the environment.

Ultimately, we want to see how well the resulting Agent can generalize based on training from human data alone.

### Action Space

The ADROIT Hand has 24 Degrees of Freedom. The full action space is documented here: https://robotics.farama.org/envs/adroit_hand/adroit_relocate/

The action space has 30 dimensions. 24 dimensions for controlling the actuators of the hand, and 6 dimenions for controlling the position of the hand in the environment.

Inputs to the hands are scaled between to fall in the range [-1, 1]

### Observation Space

The observation space for this system has 39 dimenions. 30 dimensions for describing the state of the arms actuator and the position / rotatation of the arm itself. The final 9 dimensions describe the different in location between the palm of the hand and the ball, the palm of the hand and the target, and finally, the ball and the target. The observation space is documented here: https://robotics.farama.org/envs/adroit_hand/adroit_relocate/

### Minari RL Dataset

We will use the *human* generated data from the D4RL dataset's Relocate Problem. This dataset has 9942 training steps split over 25 training episodes. This data will be used to train the Decision Transformer using an "offline" training approach. We will then evaluate the resulting Decision Transformer on corresponding environment.

In [9]:
dataset = minari.load_dataset('D4RL/relocate/human-v2', download=False)


In [10]:
env  = dataset.recover_environment()

## Training

First we'll need to set some configuration parameters for the DecisionTransformer and its underlying GPT style Transformer:

In [ ]:
config = {
    # --- Environment Dimensions ---
    "state_dim": 39,       # Defined by the ADROIT Hand Relocate observation space
    "action_dim": 30,      # Defined by the ADROIT Hand Relocate action space
    "max_ep_length": 1000, # Maximum timesteps in an episode
    # --- Transformer Architecture ---
    "embed_dim": 128,      
    "layers_num": 3,
    
    # Note: heads_num * head_size must exactly equal embed_dim. 
    # Since 128 is not divisible by 6, I changed heads_num to 4.
    "heads_num": 4,        
    "head_size": 32,       # 128 // 4 = 32
    # --- Regularization & Attention ---
    "use_bias": True,
    "dropout_rate": 0.1,   # 0.1 is standard for transformers
    
    # context_size is the max sequence length passed to the transformer.
    # In Decision Transformers, each timestep uses 3 tokens (Return, State, Action).
    # If your context window (K) of past timesteps is 20, context_size should be 20 * 3 = 60.
    "context_size": 60     
}